<a href="https://colab.research.google.com/github/Shturman71/Colab/blob/KARAOKE_LIGHT/Karaokelight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
# Шаг 1: Подготовка среды
# Запустите эту ячейку, чтобы установить необходимые библиотеки и системные пакеты.
# Установка библиотек для видео, звука и нейронных голосов
!pip install edge-tts pydub moviepy
# Установка ImageMagick для работы с текстом в видео
!apt-get install -y imagemagick

# --- Устранение проблемы с политикой ImageMagick ---
# Colab может сбрасывать системные настройки, поэтому более надежный способ:
import shutil
import tempfile

IMAGEMAGICK_POLICY_PATH = "/etc/ImageMagick-6/policy.xml"

# Создаем временную папку для модифицированного файла политики
with tempfile.TemporaryDirectory() as tmpdir:
    temp_policy_path = os.path.join(tmpdir, "policy.xml")
    shutil.copy(IMAGEMAGICK_POLICY_PATH, temp_policy_path)

    # Модифицируем скопированный файл политики
    !sudo sed -i 's/rights="none"/rights="all"/g' {temp_policy_path}

    # Устанавливаем переменную окружения, чтобы ImageMagick использовал наш модифицированный файл
    os.environ["MAGICK_POLICY_FILE"] = temp_policy_path
    print(f"ImageMagick policy file modified and set to: {temp_policy_path}")

# Дополнительная настройка MoviePy для явного указания пути к ImageMagick
from moviepy.config import change_settings
change_settings({"IMAGEMAGICK_BINARY": "/usr/bin/convert"})

# Шаг 2: Полный скрипт создания простого караоке
import asyncio
import edge_tts
from pydub import AudioSegment
from moviepy.editor import TextClip, AudioFileClip, ColorClip, CompositeVideoClip, concatenate_videoclips
import os
import re
import shutil
from google.colab import files

# --- НАСТРОЙКИ ---
# Голоса для озвучки
VOICE_RU = "ru-RU-DmitryNeural" # Дмитрий - русский голос
VOICE_EN = "en-US-GuyNeural"    # Гай - английский голос

OUTPUT_FOLDER = "simple_karaoke"
EN_FILE = "original_en.txt"       # Английский текст из источников [2-5]
RU_FILE = "translated_ru.txt"     # Ваш перевод из источников [1, 6-13]
BACKGROUND_MUSIC_FILE = "background.mp3" # Файл фоновой музыки
PAUSE_DURATION = 10.0 # @param {type:"number"} # Длительность паузы между текстами в секундах. Измените это значение.
BACKGROUND_MUSIC_VOLUME = 0.1 # Громкость фоновой музыки (от 0.0 до 1.0)
GENERATE_AUDIO_LANGUAGES = "both" # @param ['ru', 'en', 'both']

# Выбор голоса для русского текста (вопроса)
VOICE_FOR_RUSSIAN_TEXT = "English Voice (Guy)" # @param ["Russian Voice (Dmitry)", "English Voice (Guy)"]
# Выбор голоса для английского текста (ответа)
VOICE_FOR_ENGLISH_TEXT = "English Voice (Guy)" # @param ["English Voice (Guy)", "Russian Voice (Dmitry)"]

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

async def generate_voice(text, voice, filename):
    """Синтез речи через edge-tts"""
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(filename)

def create_text_video(text, audio_path, color):
    """Создание видео-фрагмента: текст на однотонном фоне"""
    audio = AudioFileClip(audio_path)
    # Создаем фон (темно-синий, как космос, который видел Ли [4])
    bg = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(audio.duration)
    # Наложение текста
    txt = TextClip(text, fontsize=60, color=color, size=(1100, None),
                   method='caption', font='Arial').set_duration(audio.duration).set_position('center')
    return CompositeVideoClip([bg, txt]).set_audio(audio)

async def main():
    # Проверка наличия ваших файлов в Colab
    if not os.path.exists(EN_FILE) or not os.path.exists(RU_FILE):
        print(f"Загрузите {EN_FILE} и {RU_FILE} в боковую панель!")
        return

    # Проверка наличия файла фоновой музыки
    if not os.path.exists(BACKGROUND_MUSIC_FILE):
        print(f"Загрузите {BACKGROUND_MUSIC_FILE} в боковую панель для фоновой музыки!")
        return

    # Загрузка фоновой музыки один раз
    background_audio_clip = AudioFileClip(BACKGROUND_MUSIC_FILE)

    # Загрузка текстов из источников
    with open(EN_FILE, 'r', encoding='utf-8') as f:
        en_sents = re.split(r'(?<=[.!?\"])\s+', f.read().replace('\n', ' ').strip())
    with open(RU_FILE, 'r', encoding='utf-8') as f:
        ru_sents = re.split(r'(?<=[.!?\"])\s+', f.read().replace('\n', ' ').strip())

    # Determine which actual voice to use based on user selection
    selected_voice_for_russian_text = VOICE_RU if VOICE_FOR_RUSSIAN_TEXT == "Russian Voice (Dmitry)" else VOICE_EN
    selected_voice_for_english_text = VOICE_EN if VOICE_FOR_ENGLISH_TEXT == "English Voice (Guy)" else VOICE_RU

    for i, (en, ru) in enumerate(zip(en_sents, ru_sents)):
        temp_ru = f"ru_{i}.mp3"
        temp_en = f"en_{i}.mp3"
        output_mp4 = f"{OUTPUT_FOLDER}/{i+1:03d}.mp4"

        # Determine which audio files will actually be generated
        audio_ru_will_be_generated = GENERATE_AUDIO_LANGUAGES in ["ru", "both"]
        audio_en_will_be_generated = GENERATE_AUDIO_LANGUAGES in ["en", "both"]

        # 1. Озвучка - Generate audio files
        audio_generation_tasks = []
        if audio_ru_will_be_generated:
            audio_generation_tasks.append(generate_voice(ru.strip(), selected_voice_for_russian_text, temp_ru))
        if audio_en_will_be_generated:
            audio_generation_tasks.append(generate_voice(en.strip(), selected_voice_for_english_text, temp_en))

        if audio_generation_tasks:
            await asyncio.gather(*audio_generation_tasks)
        else:
            print(f"Предупреждение: Не выбрано ни одной опции для генерации аудио для пары {i+1}. Продолжаю без аудио для этой пары.")


        # Создаем клип для паузы с фоновой музыкой
        # Убедимся, что фоновая аудиозапись достаточно длинная или зациклена.
        # Для простоты возьмем сегмент фоновой аудиозаписи и уменьшим громкость.
        bg_music_segment = background_audio_clip.subclip(0, PAUSE_DURATION).volumex(BACKGROUND_MUSIC_VOLUME)
        pause_clip = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(PAUSE_DURATION)
        pause_clip = pause_clip.set_audio(bg_music_segment)

        # 2. Создание видео-частей и их сборка
        clips_to_assemble = []

        # Process Question part (RU text content)
        if audio_ru_will_be_generated:
            clip_question_video = create_text_video(ru.strip(), temp_ru, 'yellow')
            clips_to_assemble.append(clip_question_video)
            clips_to_assemble.append(pause_clip) # Pause after the question

        # Process Answer part (EN text content)
        if audio_en_will_be_generated:
            clip_answer_video = create_text_video(en.strip(), temp_en, 'cyan')
            clips_to_assemble.append(clip_answer_video)
            clips_to_assemble.append(pause_clip) # Pause after the answer

        # Fallback if no audio/video segments were created for this pair
        if not clips_to_assemble:
            final_video = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(PAUSE_DURATION)
        else:
            final_video = concatenate_videoclips(clips_to_assemble)

        final_video.write_videofile(output_mp4, fps=24, codec='libx264', audio_codec='aac')

        # Очистка временных аудио
        if audio_ru_will_be_generated and os.path.exists(temp_ru):
            os.remove(temp_ru)
        if audio_en_will_be_generated and os.path.exists(temp_en):
            os.remove(temp_en)
        print(f"Видео {i+1:03d} создано.")

    # Шаг 3: Архивация и автоматическое скачивание
    shutil.make_archive('karaoke_simple', 'zip', OUTPUT_FOLDER)
    files.download('karaoke_simple.zip')
    print("Архив готов к скачиванию!")

# Запуск в Colab
await main()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
ImageMagick policy file modified and set to: /tmp/tmpdv2jz7z8/policy.xml




t:  48%|████▊     | 163/338 [09:50<00:03, 56.11it/s, now=None]

t:  48%|████▊     | 163/338 [09:50<00:03, 56.11it/s, now=None]

Moviepy - Building video simple_karaoke/001.mp4.
MoviePy - Writing audio in 001TEMP_MPY_wvf_snd.mp4




t:  48%|████▊     | 163/338 [09:52<00:03, 56.11it/s, now=None]

t:  48%|████▊     | 163/338 [09:52<00:03, 56.11it/s, now=None]

MoviePy - Done.
Moviepy - Writing video simple_karaoke/001.mp4



...



t:  48%|████▊     | 163/338 [18:33<00:03, 56.11it/s, now=None]

t:  48%|████▊     | 163/338 [18:33<00:03, 56.11it/s, now=None]

Moviepy - Done !
Moviepy - video ready simple_karaoke/031.mp4
Видео 031 создано.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Архив готов к скачиванию!


In [19]:
from moviepy.editor import VideoFileClip, concatenate_videoclips
import os

# --- Объединение видео ---

final_video_output = "final_karaoke_video.mp4"

# Получить список всех сгенерированных видеофайлов
video_files = sorted([os.path.join(OUTPUT_FOLDER, f) for f in os.listdir(OUTPUT_FOLDER) if f.endswith('.mp4')])

if not video_files:
    print(f"Ошибка: В папке '{OUTPUT_FOLDER}' не найдено MP4-видеофайлов для объединения.")
else:
    print(f"Объединяю {len(video_files)} видеофайлов...")
    # Загрузить все видеоклипы
    clips = [VideoFileClip(f) for f in video_files]

    # Объединить их
    final_clip = concatenate_videoclips(clips)

    # Записать финальное видео
    final_clip.write_videofile(final_video_output, fps=24, codec='libx264', audio_codec='aac')

    print(f"Все видеофайлы объединены в '{final_video_output}'.")

    # Опционально: Скачать объединенное видео
    from google.colab import files
    files.download(final_video_output)
    print(f"Файл '{final_video_output}' готов к скачиванию.")

Объединяю 28 видеофайлов...


KeyboardInterrupt: 

In [29]:
from moviepy.editor import VideoFileClip, concatenate_videoclips
import os

# --- Объединение видео ---

final_video_output = "final_karaoke_video.mp4"

# Получить список всех сгенерированных видеофайлов
video_files = sorted([os.path.join(OUTPUT_FOLDER, f) for f in os.listdir(OUTPUT_FOLDER) if f.endswith('.mp4')])

if not video_files:
    print(f"Ошибка: В папке '{OUTPUT_FOLDER}' не найдено MP4-видеофайлов для объединения.")
else:
    print(f"Объединяю {len(video_files)} видеофайлов...")
    # Загрузить все видеоклипы
    clips = [VideoFileClip(f) for f in video_files]

    # Объединить их
    final_clip = concatenate_videoclips(clips)

    # Записать финальное видео
    final_clip.write_videofile(final_video_output, fps=24, codec='libx264', audio_codec='aac')

    print(f"Все видеофайлы объединены в '{final_video_output}'.")

    # Опционально: Скачать объединенное видео
    from google.colab import files
    files.download(final_video_output)
    print(f"Файл '{final_video_output}' готов к скачиванию.")

Объединяю 31 видеофайлов...


KeyboardInterrupt: 

In [21]:
import pandas as pd
import os

QA_FILE = 'qa_data.csv'
EN_OUTPUT_FILE = 'original_en.txt'
RU_OUTPUT_FILE = 'translated_ru.txt'

if os.path.exists(QA_FILE):
    print(f"Файл '{QA_FILE}' найден. Обрабатываю...")
    try:
        # First attempt: try reading with semicolon as separator.
        # This handles cases where 'question;answer' is not quoted in the header.
        df_qa = pd.read_csv(QA_FILE, sep=';')
        print(f"После pd.read_csv(QA_FILE, sep=';'), столбцы: {df_qa.columns.tolist()}")

        # Normalize column names to lowercase for robust checking
        df_qa.columns = [col.lower() for col in df_qa.columns]

        if 'question' in df_qa.columns and 'answer' in df_qa.columns and len(df_qa.columns) == 2:
            # If successfully read as two distinct columns (case-insensitive)
            print("Столбцы 'question' и 'answer' успешно обнаружены.")
        elif len(df_qa.columns) == 1 and df_qa.columns[0] == 'question;answer':
            # If it's a single column named 'question;answer' (e.g., if the header was quoted like "question;answer")
            print("Обнаружен единственный столбец 'question;answer'. Попытка разделения его содержимого.")
            # Split the content of the single column into two new columns
            # n=1 ensures that only the first semicolon is used for splitting, preventing issues if
            # questions or answers contain semicolons themselves.
            split_data = df_qa[df_qa.columns[0]].str.split(';', n=1, expand=True)

            if split_data.shape[1] == 2:
                df_qa['question'] = split_data[0].str.strip() # Strip whitespace
                df_qa['answer'] = split_data[1].str.strip()   # Strip whitespace
                df_qa = df_qa[['question', 'answer']] # Keep only the newly created columns
                print("Столбцы 'question' и 'answer' успешно созданы путем разделения содержимого.")
            else:
                print(f"Ошибка: Разделение содержимого столбца '{df_qa.columns[0]}' по символу ';' не дало двух частей. Проверьте формат данных в CSV.")
                print(f"Пример данных в столбце '{df_qa.columns[0]}' (первые 5 строк):\n{df_qa[df_qa.columns[0]].head().tolist()}")
                raise ValueError("Не удалось разделить объединенный столбец на 'question' и 'answer'.")
        else:
            # If neither of the above conditions met, it's an unexpected format
            print(f"Ошибка: Неожиданный формат CSV-файла. Ожидались столбцы 'question' и 'answer' или один столбец 'question;answer'. Текущие столбцы: {df_qa.columns.tolist()}")
            raise ValueError("Некорректный формат CSV-файла: отсутствуют ожидаемые столбцы.")

    except Exception as e:
        print(f"Критическая ошибка при чтении или обработке файла '{QA_FILE}': {e}")
        print("Пожалуйста, убедитесь, что ваш файл qa_data.csv корректно отформатирован.")
        exit() # Terminate execution if a critical error occurs

    # Final check after all processing
    if 'question' in df_qa.columns and 'answer' in df_qa.columns:
        print(f"Финальная проверка: Столбцы 'question' и 'answer' присутствуют. Продолжаю.")
        # Записываем ответы в original_en.txt (обратный порядок)
        with open(EN_OUTPUT_FILE, 'w', encoding='utf-8') as f_en:
            for answer in df_qa['answer']:
                f_en.write(str(answer).strip() + '\n')

        # Записываем вопросы в translated_ru.txt (обратный порядок)
        with open(RU_OUTPUT_FILE, 'w', encoding='utf-8') as f_ru:
            for question in df_qa['question']:
                f_ru.write(str(question).strip() + '\n')

        print(f"Файлы '{EN_OUTPUT_FILE}' и '{RU_OUTPUT_FILE}' успешно созданы из '{QA_FILE}' (содержимое поменялось местами).")
    else:
        # This branch should ideally not be reached if the try-except block is robust
        print(f"Ошибка: Файл '{QA_FILE}' должен содержать столбцы 'question' и 'answer' после обработки. Текущие столбцы: {df_qa.columns.tolist()}")
else:
    print(f"Ошибка: Файл '{QA_FILE}' не найден в текущей директории. Пожалуйста, загрузите его.")

Файл 'qa_data.csv' найден. Обрабатываю...
После pd.read_csv(QA_FILE, sep=';'), столбцы: ['Question;Answer']
Обнаружен единственный столбец 'question;answer'. Попытка разделения его содержимого.
Столбцы 'question' и 'answer' успешно созданы путем разделения содержимого.
Финальная проверка: Столбцы 'question' и 'answer' присутствуют. Продолжаю.
Файлы 'original_en.txt' и 'translated_ru.txt' успешно созданы из 'qa_data.csv' (содержимое поменялось местами).


# Новый раздел

In [30]:
# Шаг 1: Подготовка среды
# Запустите эту ячейку, чтобы установить необходимые библиотеки и системные пакеты.
# Установка библиотек для видео, звука и нейронных голосов
!pip install edge-tts pydub moviepy
# Установка ImageMagick для работы с текстом в видео
!apt-get install -y imagemagick
# Изменение политики ImageMagick для разрешения создания временных файлов
# Эта команда пытается установить все права доступа для всех политик ImageMagick.
# Это должно решить проблему с 'security policy'.
!sudo sed -i 's/rights="none"/rights="all"/g' /etc/ImageMagick-6/policy.xml
# Шаг 2: Полный скрипт создания простого караоке
import asyncio
import edge_tts
from pydub import AudioSegment
from moviepy.editor import TextClip, AudioFileClip, ColorClip, CompositeVideoClip, concatenate_videoclips
import os
import re
import shutil
from google.colab import files

# --- НАСТРОЙКИ ---
# Голоса для озвучки приключений Ли и пришельца [3]
VOICE_RU = "ru-RU-DmitryNeural"
VOICE_EN = "en-US-GuyNeural"
OUTPUT_FOLDER = "simple_karaoke"
EN_FILE = "original_en.txt"       # Английский текст из источников [2-5]
RU_FILE = "translated_ru.txt"     # Ваш перевод из источников [1, 6-13]
BACKGROUND_MUSIC_FILE = "background.mp3" # Файл фоновой музыки
PAUSE_DURATION = 10.0 # Длительность паузы между текстами в секундах
BACKGROUND_MUSIC_VOLUME = 0.1 # Громкость фоновой музыки (от 0.0 до 1.0)

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

async def generate_voice(text, voice, filename):
    """Синтез речи через edge-tts"""
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(filename)

def create_text_video(text, audio_path, color):
    """Создание видео-фрагмента: текст на однотонном фоне"""
    audio = AudioFileClip(audio_path)
    # Создаем фон (темно-синий, как космос, который видел Ли [4])
    bg = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(audio.duration)
    # Наложение текста
    txt = TextClip(text, fontsize=60, color=color, size=(1100, None),
                   method='caption', font='Arial').set_duration(audio.duration).set_position('center')
    return CompositeVideoClip([bg, txt]).set_audio(audio)

async def main():
    # Проверка наличия ваших файлов в Colab
    if not os.path.exists(EN_FILE) or not os.path.exists(RU_FILE):
        print(f"Загрузите {EN_FILE} и {RU_FILE} в боковую панель!")
        return

    # Проверка наличия файла фоновой музыки
    if not os.path.exists(BACKGROUND_MUSIC_FILE):
        print(f"Загрузите {BACKGROUND_MUSIC_FILE} в боковую панель для фоновой музыки!")
        return

    # Загрузка фоновой музыки один раз
    background_audio_clip = AudioFileClip(BACKGROUND_MUSIC_FILE)

    # Загрузка текстов из источников
    with open(EN_FILE, 'r', encoding='utf-8') as f:
        en_sents = re.split(r'(?<=[.!?"])\s+', f.read().replace('\n', ' ').strip())
    with open(RU_FILE, 'r', encoding='utf-8') as f:
        ru_sents = re.split(r'(?<=[.!?"])\s+', f.read().replace('\n', ' ').strip())

    for i, (en, ru) in enumerate(zip(en_sents, ru_sents)):
        temp_ru = f"ru_{i}.mp3"
        temp_en = f"en_{i}.mp3"
        output_mp4 = f"{OUTPUT_FOLDER}/{i+1:03d}.mp4"

        # 1. Озвучка
        await asyncio.gather(
            generate_voice(ru.strip(), VOICE_RU, temp_ru),
            generate_voice(en.strip(), VOICE_EN, temp_en)
        )

        # 2. Создание видео-частей
        # Желтый для перевода (например, "Они усердно трудились" [1])
        clip_ru = create_text_video(ru.strip(), temp_ru, 'yellow')
        # Голубой для оригинала (например, "They had worked hard" [2])
        clip_en = create_text_video(en.strip(), temp_en, 'cyan')

        # Создаем клип для паузы с фоновой музыкой
        # Убедимся, что фоновая аудиозапись достаточно длинная или зациклена.
        # Для простоты возьмем сегмент фоновой аудиозаписи и уменьшим громкость.
        bg_music_segment = background_audio_clip.subclip(0, PAUSE_DURATION).volumex(BACKGROUND_MUSIC_VOLUME)
        pause_clip = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(PAUSE_DURATION)
        pause_clip = pause_clip.set_audio(bg_music_segment)

        # 3. Сборка финального видео для одного предложения
        final_video = concatenate_videoclips([clip_ru, pause_clip, clip_en])
        final_video.write_videofile(output_mp4, fps=24, codec='libx264', audio_codec='aac')

        # Очистка временных аудио
        os.remove(temp_ru)
        os.remove(temp_en)
        print(f"Видео {i+1:03d} создано.")

    # Шаг 3: Архивация и автоматическое скачивание
    shutil.make_archive('karaoke_simple', 'zip', OUTPUT_FOLDER)
    files.download('karaoke_simple.zip')
    print("Архив готов к скачиванию!")

# Запуск в Colab
await main()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
Moviepy - Building video simple_karaoke/001.mp4.
MoviePy - Writing audio in 001TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video simple_karaoke/001.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/001.mp4
Видео 001 создано.
Moviepy - Building video simple_karaoke/002.mp4.
MoviePy - Writing audio in 002TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video simple_karaoke/002.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/002.mp4
Видео 002 создано.
Moviepy - Building video simple_karaoke/003.mp4.
MoviePy - Writing audio in 003TEMP_MPY_wvf_snd.mp4


...

MoviePy - Done.
Moviepy - Writing video simple_karaoke/031.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/031.mp4
Видео 031 создано.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Архив готов к скачиванию!


In [31]:
# Шаг 1: Подготовка среды
# Запустите эту ячейку, чтобы установить необходимые библиотеки и системные пакеты.
# Установка библиотек для видео, звука и нейронных голосов
!pip install edge-tts pydub moviepy
# Установка ImageMagick для работы с текстом в видео
!apt-get install -y imagemagick
# Изменение политики ImageMagick для разрешения создания временных файлов
# Эта команда пытается установить все права доступа для всех политик ImageMagick.
# Это должно решить проблему с 'security policy'.
!sudo sed -i 's/rights="none"/rights="all"/g' /etc/ImageMagick-6/policy.xml
# Шаг 2: Полный скрипт создания простого караоке
import asyncio
import edge_tts
from pydub import AudioSegment
from moviepy.editor import TextClip, AudioFileClip, ColorClip, CompositeVideoClip, concatenate_videoclips
import os
import re
import shutil
from google.colab import files

# --- НАСТРОЙКИ ---
# Голоса для озвучки приключений Ли и пришельца [3]
VOICE_RU = "ru-RU-DmitryNeural"
VOICE_EN = "en-US-GuyNeural"
OUTPUT_FOLDER = "simple_karaoke"
EN_FILE = "original_en.txt"       # Английский текст из источников [2-5]
RU_FILE = "translated_ru.txt"     # Ваш перевод из источников [1, 6-13]
BACKGROUND_MUSIC_FILE = "background.mp3" # Файл фоновой музыки
PAUSE_DURATION = 10.0 # Длительность паузы между текстами в секундах
BACKGROUND_MUSIC_VOLUME = 0.1 # Громкость фоновой музыки (от 0.0 до 1.0)

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

async def generate_voice(text, voice, filename):
    """Синтез речи через edge-tts"""
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(filename)

def create_text_video(text, audio_path, color):
    """Создание видео-фрагмента: текст на однотонном фоне"""
    audio = AudioFileClip(audio_path)
    # Создаем фон (темно-синий, как космос, который видел Ли [4])
    bg = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(audio.duration)
    # Наложение текста
    txt = TextClip(text, fontsize=60, color=color, size=(1100, None),
                   method='caption', font='Arial').set_duration(audio.duration).set_position('center')
    return CompositeVideoClip([bg, txt]).set_audio(audio)

async def main():
    # Проверка наличия ваших файлов в Colab
    if not os.path.exists(EN_FILE) or not os.path.exists(RU_FILE):
        print(f"Загрузите {EN_FILE} и {RU_FILE} в боковую панель!")
        return

    # Проверка наличия файла фоновой музыки
    if not os.path.exists(BACKGROUND_MUSIC_FILE):
        print(f"Загрузите {BACKGROUND_MUSIC_FILE} в боковую панель для фоновой музыки!")
        return

    # Загрузка фоновой музыки один раз
    background_audio_clip = AudioFileClip(BACKGROUND_MUSIC_FILE)

    # Загрузка текстов из источников
    with open(EN_FILE, 'r', encoding='utf-8') as f:
        en_sents = re.split(r'(?<=[.!?"])\s+', f.read().replace('\n', ' ').strip())
    with open(RU_FILE, 'r', encoding='utf-8') as f:
        ru_sents = re.split(r'(?<=[.!?"])\s+', f.read().replace('\n', ' ').strip())

    for i, (en, ru) in enumerate(zip(en_sents, ru_sents)):
        temp_ru = f"ru_{i}.mp3"
        temp_en = f"en_{i}.mp3"
        output_mp4 = f"{OUTPUT_FOLDER}/{i+1:03d}.mp4"

        # 1. Озвучка
        await asyncio.gather(
           # generate_voice(ru.strip(), VOICE_RU, temp_ru),

            generate_voice(en.strip(), VOICE_EN, temp_en)
        )

        # 2. Создание видео-частей
        # Желтый для перевода (например, "Они усердно трудились" [1])
        clip_ru = create_text_video(ru.strip(), temp_ru, 'yellow')
        # Голубой для оригинала (например, "They had worked hard" [2])
        clip_en = create_text_video(en.strip(), temp_en, 'cyan')

        # Создаем клип для паузы с фоновой музыкой
        # Убедимся, что фоновая аудиозапись достаточно длинная или зациклена.
        # Для простоты возьмем сегмент фоновой аудиозаписи и уменьшим громкость.
        bg_music_segment = background_audio_clip.subclip(0, PAUSE_DURATION).volumex(BACKGROUND_MUSIC_VOLUME)
        pause_clip = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(PAUSE_DURATION)
        pause_clip = pause_clip.set_audio(bg_music_segment)

        # 3. Сборка финального видео для одного предложения
        final_video = concatenate_videoclips([clip_ru, pause_clip, clip_en])
        final_video.write_videofile(output_mp4, fps=24, codec='libx264', audio_codec='aac')

        # Очистка временных аудио
        os.remove(temp_ru)
        os.remove(temp_en)
        print(f"Видео {i+1:03d} создано.")

    # Шаг 3: Архивация и автоматическое скачивание
    shutil.make_archive('karaoke_simple', 'zip', OUTPUT_FOLDER)
    files.download('karaoke_simple.zip')
    print("Архив готов к скачиванию!")

# Запуск в Colab
await main()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


OSError: MoviePy error: the file ru_0.mp3 could not be found!
Please check that you entered the correct path.

In [ ]:
# Шаг 1: Подготовка среды
# Запустите эту ячейку, чтобы установить необходимые библиотеки и системные пакеты.
# Установка библиотек для видео, звука и нейронных голосов
!pip install edge-tts pydub moviepy
# Установка ImageMagick для работы с текстом в видео
!apt-get install -y imagemagick
# Изменение политики ImageMagick для разрешения создания временных файлов
# Эта команда пытается установить все права доступа для всех политик ImageMagick.
# Это должно решить проблему с 'security policy'.
!sudo sed -i 's/rights="none"/rights="all"/g' /etc/ImageMagick-6/policy.xml
# Шаг 2: Полный скрипт создания простого караоке
import asyncio
import edge_tts
from pydub import AudioSegment
from moviepy.editor import TextClip, AudioFileClip, ColorClip, CompositeVideoClip, concatenate_videoclips
import os
import re
import shutil
from google.colab import files

# --- НАСТРОЙКИ ---
# Голоса для озвучки приключений Ли и пришельца [3]
VOICE_RU = "ru-RU-DmitryNeural"
VOICE_EN = "en-US-GuyNeural"
OUTPUT_FOLDER = "simple_karaoke"
EN_FILE = "original_en.txt"       # Английский текст из источников [2-5]
RU_FILE = "translated_ru.txt"     # Ваш перевод из источников [1, 6-13]
BACKGROUND_MUSIC_FILE = "background.mp3" # Файл фоновой музыки
PAUSE_DURATION = 10.0 # Длительность паузы между текстами в секундах
BACKGROUND_MUSIC_VOLUME = 0.1 # Громкость фоновой музыки (от 0.0 до 1.0)

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

async def generate_voice(text, voice, filename):
    """Синтез речи через edge-tts"""
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(filename)

def create_text_video(text, audio_path, color):
    """Создание видео-фрагмента: текст на однотонном фоне"""
    audio = AudioFileClip(audio_path)
    # Создаем фон (темно-синий, как космос, который видел Ли [4])
    bg = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(audio.duration)
    # Наложение текста
    txt = TextClip(text, fontsize=60, color=color, size=(1100, None),
                   method='caption', font='Arial').set_duration(audio.duration).set_position('center')
    return CompositeVideoClip([bg, txt]).set_audio(audio)

async def main():
    # Проверка наличия ваших файлов в Colab
    if not os.path.exists(EN_FILE) or not os.path.exists(RU_FILE):
        print(f"Загрузите {EN_FILE} и {RU_FILE} в боковую панель!")
        return

    # Проверка наличия файла фоновой музыки
    if not os.path.exists(BACKGROUND_MUSIC_FILE):
        print(f"Загрузите {BACKGROUND_MUSIC_FILE} в боковую панель для фоновой музыки!")
        return

    # Загрузка фоновой музыки один раз
    background_audio_clip = AudioFileClip(BACKGROUND_MUSIC_FILE)

    # Загрузка текстов из источников
    with open(EN_FILE, 'r', encoding='utf-8') as f:
        en_sents = re.split(r'(?<=[.!?"])\s+', f.read().replace('\n', ' ').strip())
    with open(RU_FILE, 'r', encoding='utf-8') as f:
        ru_sents = re.split(r'(?<=[.!?"])\s+', f.read().replace('\n', ' ').strip())

    for i, (en, ru) in enumerate(zip(en_sents, ru_sents)):
        temp_ru = f"ru_{i}.mp3"
        temp_en = f"en_{i}.mp3"
        output_mp4 = f"{OUTPUT_FOLDER}/{i+1:03d}.mp4"

        # 1. Озвучка
        await asyncio.gather(
           # generate_voice(ru.strip(), VOICE_RU, temp_ru),
            generate_voice(en.strip(), VOICE_EN, temp_en)
        )

        # 2. Создание видео-частей
        # Желтый для перевода (например, "Они усердно трудились" [1])
        clip_ru = create_text_video(ru.strip(), temp_ru, 'yellow')
        # Голубой для оригинала (например, "They had worked hard" [2])
        clip_en = create_text_video(en.strip(), temp_en, 'cyan')

        # Создаем клип для паузы с фоновой музыкой
        # Убедимся, что фоновая аудиозапись достаточно длинная или зациклена.
        # Для простоты возьмем сегмент фоновой аудиозаписи и уменьшим громкость.
        bg_music_segment = background_audio_clip.subclip(0, PAUSE_DURATION).volumex(BACKGROUND_MUSIC_VOLUME)
        pause_clip = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(PAUSE_DURATION)
        pause_clip = pause_clip.set_audio(bg_music_segment)

        # 3. Сборка финального видео для одного предложения
        final_video = concatenate_videoclips([clip_ru, pause_clip, clip_en])
        final_video.write_videofile(output_mp4, fps=24, codec='libx264', audio_codec='aac')

        # Очистка временных аудио
        os.remove(temp_ru)
        os.remove(temp_en)
        print(f"Видео {i+1:03d} создано.")

    # Шаг 3: Архивация и автоматическое скачивание
    shutil.make_archive('karaoke_simple', 'zip', OUTPUT_FOLDER)
    files.download('karaoke_simple.zip')
    print("Архив готов к скачиванию!")

# Запуск в Colab
await main()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
Moviepy - Building video simple_karaoke/001.mp4.
MoviePy - Writing audio in 001TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video simple_karaoke/001.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/001.mp4
Видео 001 создано.
Moviepy - Building video simple_karaoke/002.mp4.
MoviePy - Writing audio in 002TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video simple_karaoke/002.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/002.mp4
Видео 002 создано.
Moviepy - Building video simple_karaoke/003.mp4.
MoviePy - Writing audio in 003TEMP_MPY_wvf_snd.mp4


...

MoviePy - Done.
Moviepy - Writing video simple_karaoke/027.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/027.mp4
Видео 027 создано.
Moviepy - Building video simple_karaoke/028.mp4.
MoviePy - Writing audio in 028TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video simple_karaoke/028.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/028.mp4
Видео 028 создано.


NoAudioReceived: No audio was received. Please verify that your parameters are correct.

In [ ]:
!edge-tts --list-voices

Name                               Gender    ContentCategories      VoicePersonalities
---------------------------------  --------  ---------------------  --------------------------------------
af-ZA-AdriNeural                   Female    General                Friendly, Positive
af-ZA-WillemNeural                 Male      General                Friendly, Positive
am-ET-AmehaNeural                  Male      General                Friendly, Positive
am-ET-MekdesNeural                 Female    General                Friendly, Positive
ar-AE-FatimaNeural                 Female    General                Friendly, Positive
ar-AE-HamdanNeural                 Male      General                Friendly, Positive
ar-BH-AliNeural                    Male      General                Friendly, Positive
ar-BH-LailaNeural                  Female    General                Friendly, Positive
ar-DZ-AminaNeural                  Female    General                Friendly, Positive
ar-DZ-IsmaelNeural     

In [ ]:
import asyncio
from IPython.display import Audio, display

# Пример фразы для синтеза
sample_ru_phrase = "Привет, это тестовая фраза на русском языке."
sample_en_phrase = "Hello, this is a test phrase in English."

output_ru_audio = "sample_ru_phrase.mp3"
output_en_audio = "sample_en_phrase.mp3"

async def synthesize_sample_phrases():
    print(f"Синтезирую русскую фразу голосом: {VOICE_RU}")
    await generate_voice(sample_ru_phrase, VOICE_RU, output_ru_audio)
    print(f"Синтезирую английскую фразу голосом: {VOICE_EN}")
    await generate_voice(sample_en_phrase, VOICE_EN, output_en_audio)
    print("Синтез завершен.")

await synthesize_sample_phrases()

print("Прослушать русскую фразу:")
display(Audio(output_ru_audio, autoplay=False))

print("Прослушать английскую фразу:")
display(Audio(output_en_audio, autoplay=False))

Синтезирую русскую фразу голосом: ru-RU-DmitryNeural
Синтезирую английскую фразу голосом: en-US-GuyNeural
Синтез завершен.
Прослушать русскую фразу:


Прослушать английскую фразу:


После того как вы выберете подходящий голос из списка, вы сможете изменить его в следующей ячейке, заменив существующие значения переменных `VOICE_RU` или `VOICE_EN`.

In [ ]:
# Пример изменения голоса (замените на свои выбранные голоса)
VOICE_RU = "ru-RU-SvetlanaNeural"  # Пример: женский голос для русского
VOICE_EN = "en-GB-RyanNeural"      # Пример: британский мужской голос для английского

print(f"Новый русский голос установлен как: {VOICE_RU}")
print(f"Новый английский голос установлен как: {VOICE_EN}")

In [ ]:
Шаг 1: Подготовка среды
Запустите эту ячейку, чтобы установить необходимые библиотеки и системные пакеты.
# Установка библиотек для видео, звука и нейронных голосов
!pip install edge-tts pydub moviepy
# Установка ImageMagick для работы с текстом в видео
!apt-get install -y imagemagick
Шаг 2: Полный скрипт создания простого караоке
import asyncio
import edge_tts
from pydub import AudioSegment
from moviepy.editor import TextClip, AudioFileClip, ColorClip, CompositeVideoClip, concatenate_videoclips
import os
import re
import shutil
from google.colab import files

# --- НАСТРОЙКИ ---
# Голоса для озвучки приключений Ли и пришельца [3]
VOICE_RU = "ru-RU-DmitryNeural"
VOICE_EN = "en-US-GuyNeural"
OUTPUT_FOLDER = "simple_karaoke"
EN_FILE = "original_en.txt"       # Английский текст из источников [2-5]
RU_FILE = "translated_ru.txt"     # Ваш перевод из источников [1, 6-13]

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

async def generate_voice(text, voice, filename):
    """Синтез речи через edge-tts"""
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(filename)

def create_text_video(text, audio_path, color):
    """Создание видео-фрагмента: текст на однотонном фоне"""
    audio = AudioFileClip(audio_path)
    # Создаем фон (темно-синий, как космос, который видел Ли [4])
    bg = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(audio.duration)
    # Наложение текста
    txt = TextClip(text, fontsize=60, color=color, size=(1100, None),
                   method='caption', font='Arial').set_duration(audio.duration).set_position('center')
    return CompositeVideoClip([bg, txt]).set_audio(audio)

async def main():
    # Проверка наличия ваших файлов в Colab
    if not os.path.exists(EN_FILE) or not os.path.exists(RU_FILE):
        print(f"Загрузите {EN_FILE} и {RU_FILE} в боковую панель!")
        return

    # Загрузка текстов из источников
    with open(EN_FILE, 'r', encoding='utf-8') as f:
        en_sents = re.split(r'(?<=[.!?])\s+', f.read().replace('\n', ' ').strip())
    with open(RU_FILE, 'r', encoding='utf-8') as f:
        ru_sents = re.split(r'(?<=[.!?])\s+', f.read().replace('\n', ' ').strip())

    for i, (en, ru) in enumerate(zip(en_sents, ru_sents)):
        temp_ru = f"ru_{i}.mp3"
        temp_en = f"en_{i}.mp3"
        output_mp4 = f"{OUTPUT_FOLDER}/{i+1:03d}.mp4"

        # 1. Озвучка
        await asyncio.gather(
            generate_voice(ru.strip(), VOICE_RU, temp_ru),
            generate_voice(en.strip(), VOICE_EN, temp_en)
        )

        # 2. Создание видео-частей
        # Желтый для перевода (например, "Они усердно трудились" [1])
        clip_ru = create_text_video(ru.strip(), temp_ru, 'yellow')
        # Голубой для оригинала (например, "They had worked hard" [2])
        clip_en = create_text_video(en.strip(), temp_en, 'cyan')

        # 3. Сборка финального видео для одного предложения
        final_video = concatenate_videoclips([clip_ru, clip_en])
        final_video.write_videofile(output_mp4, fps=24, codec='libx264', audio_codec='aac')

        # Очистка временных аудио
        os.remove(temp_ru)
        os.remove(temp_en)
        print(f"Видео {i+1:03d} создано.")

    # Шаг 3: Архивация и автоматическое скачивание
    shutil.make_archive('karaoke_simple', 'zip', OUTPUT_FOLDER)
    files.download('karaoke_simple.zip')
    print("Архив готов к скачиванию!")

# Запуск в Colab
await main()

In [ ]:
 Шаг 1: Подготовка среды
Запустите эту ячейку, чтобы установить необходимые библиотеки и системные пакеты.
# Установка библиотек для видео, звука и нейронных голосов
!pip install edge-tts pydub moviepy
# Установка ImageMagick для работы с текстом в видео
!apt-get install -y imagemagick
Шаг 2: Полный скрипт создания простого караоке
import asyncio
import edge_tts
from pydub import AudioSegment
from moviepy.editor import TextClip, AudioFileClip, ColorClip, CompositeVideoClip, concatenate_videoclips
import os
import re
import shutil
from google.colab import files

# --- НАСТРОЙКИ ---
# Голоса для озвучки приключений Ли и пришельца [3]
VOICE_RU = "ru-RU-DmitryNeural"
VOICE_EN = "en-US-GuyNeural"
OUTPUT_FOLDER = "simple_karaoke"
EN_FILE = "original_en.txt"       # Английский текст из источников [2-5]
RU_FILE = "translated_ru.txt"     # Ваш перевод из источников [1, 6-13]

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

async def generate_voice(text, voice, filename):
    """Синтез речи через edge-tts"""
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(filename)

def create_text_video(text, audio_path, color):
    """Создание видео-фрагмента: текст на однотонном фоне"""
    audio = AudioFileClip(audio_path)
    # Создаем фон (темно-синий, как космос, который видел Ли [4])
    bg = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(audio.duration)
    # Наложение текста
    txt = TextClip(text, fontsize=60, color=color, size=(1100, None),
                   method='caption', font='Arial').set_duration(audio.duration).set_position('center')
    return CompositeVideoClip([bg, txt]).set_audio(audio)

async def main():
    # Проверка наличия ваших файлов в Colab
    if not os.path.exists(EN_FILE) or not os.path.exists(RU_FILE):
        print(f"Загрузите {EN_FILE} и {RU_FILE} в боковую панель!")
        return

    # Загрузка текстов из источников
    with open(EN_FILE, 'r', encoding='utf-8') as f:
        en_sents = re.split(r'(?<=[.!?])\s+', f.read().replace('\n', ' ').strip())
    with open(RU_FILE, 'r', encoding='utf-8') as f:
        ru_sents = re.split(r'(?<=[.!?])\s+', f.read().replace('\n', ' ').strip())

    for i, (en, ru) in enumerate(zip(en_sents, ru_sents)):
        temp_ru = f"ru_{i}.mp3"
        temp_en = f"en_{i}.mp3"
        output_mp4 = f"{OUTPUT_FOLDER}/{i+1:03d}.mp4"

        # 1. Озвучка
        await asyncio.gather(
           # generate_voice(ru.strip(), VOICE_RU, temp_ru),

            generate_voice(en.strip(), VOICE_EN, temp_en)
        )

        # 2. Создание видео-частей
        # Желтый для перевода (например, "Они усердно трудились" [1])
        clip_ru = create_text_video(ru.strip(), temp_ru, 'yellow')
        # Голубой для оригинала (например, "They had worked hard" [2])
        clip_en = create_text_video(en.strip(), temp_en, 'cyan')

        # 3. Сборка финального видео для одного предложения
        final_video = concatenate_videoclips([clip_ru, clip_en])
        final_video.write_videofile(output_mp4, fps=24, codec='libx264', audio_codec='aac')

        # Очистка временных аудио
        os.remove(temp_ru)
        os.remove(temp_en)
        print(f"Видео {i+1:03d} создано.")

    # Шаг 3: Архивация и автоматическое скачивание
    shutil.make_archive('karaoke_simple', 'zip', OUTPUT_FOLDER)
    files.download('karaoke_simple.zip')
    print("Архив готов к скачиванию!")

# Запуск в Colab
await main()

In [25]:
 Шаг 1: Подготовка среды
Запустите эту ячейку, чтобы установить необходимые библиотеки и системные пакеты.
# Установка библиотек для видео, звука и нейронных голосов
!pip install edge-tts pydub moviepy
# Установка ImageMagick для работы с текстом в видео
!apt-get install -y imagemagick
Шаг 2: Полный скрипт создания простого караоке
import asyncio
import edge_tts
from pydub import AudioSegment
from moviepy.editor import TextClip, AudioFileClip, ColorClip, CompositeVideoClip, concatenate_videoclips
import os
import re
import shutil
from google.colab import files

# --- НАСТРОЙКИ ---
# Голоса для озвучки приключений Ли и пришельца [3]
VOICE_RU = "ru-RU-DmitryNeural"
VOICE_EN = "en-US-GuyNeural"
OUTPUT_FOLDER = "simple_karaoke"
EN_FILE = "original_en.txt"       # Английский текст из источников [2-5]
RU_FILE = "translated_ru.txt"     # Ваш перевод из источников [1, 6-13]

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

async def generate_voice(text, voice, filename):
    """Синтез речи через edge-tts"""
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(filename)

def create_text_video(text, audio_path, color):
    """Создание видео-фрагмента: текст на однотонном фоне"""
    audio = AudioFileClip(audio_path)
    # Создаем фон (темно-синий, как космос, который видел Ли [4])
    bg = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(audio.duration)
    # Наложение текста
    txt = TextClip(text, fontsize=60, color=color, size=(1100, None),
                   method='caption', font='Arial').set_duration(audio.duration).set_position('center')
    return CompositeVideoClip([bg, txt]).set_audio(audio)

async def main():
    # Проверка наличия ваших файлов в Colab
    if not os.path.exists(EN_FILE) or not os.path.exists(RU_FILE):
        print(f"Загрузите {EN_FILE} и {RU_FILE} в боковую панель!")
        return

    # Загрузка текстов из источников
    with open(EN_FILE, 'r', encoding='utf-8') as f:
        en_sents = re.split(r'(?<=[.!?])\s+', f.read().replace('\n', ' ').strip())
    with open(RU_FILE, 'r', encoding='utf-8') as f:
        ru_sents = re.split(r'(?<=[.!?])\s+', f.read().replace('\n', ' ').strip())

    for i, (en, ru) in enumerate(zip(en_sents, ru_sents)):
        temp_ru = f"ru_{i}.mp3"
        temp_en = f"en_{i}.mp3"
        output_mp4 = f"{OUTPUT_FOLDER}/{i+1:03d}.mp4"

        # 1. Озвучка
        await asyncio.gather(
           # generate_voice(ru.strip(), VOICE_RU, temp_ru),

            generate_voice(en.strip(), VOICE_EN, temp_en)
        )

        # 2. Создание видео-частей
        # Желтый для перевода (например, "Они усердно трудились" [1])
        clip_ru = create_text_video(ru.strip(), temp_ru, 'yellow')
        # Голубой для оригинала (например, "They had worked hard" [2])
        clip_en = create_text_video(en.strip(), temp_en, 'cyan')

        # 3. Сборка финального видео для одного предложения
        final_video = concatenate_videoclips([clip_ru, clip_en])
        final_video.write_videofile(output_mp4, fps=24, codec='libx264', audio_codec='aac')

        # Очистка временных аудио
        os.remove(temp_ru)
        os.remove(temp_en)
        print(f"Видео {i+1:03d} создано.")

    # Шаг 3: Архивация и автоматическое скачивание
    shutil.make_archive('karaoke_simple', 'zip', OUTPUT_FOLDER)
    files.download('karaoke_simple.zip')
    print("Архив готов к скачиванию!")

# Запуск в Colab
await main()

SyntaxError: invalid syntax (2615001408.py, line 1)

In [23]:
# Шаг 1: Подготовка среды
# Запустите эту ячейку, чтобы установить необходимые библиотеки и системные пакеты.
# Установка библиотек для видео, звука и нейронных голосов
!pip install edge-tts pydub moviepy
# Установка ImageMagick для работы с текстом в видео
!apt-get install -y imagemagick
# Шаг 2: Полный скрипт создания простого караоке
import asyncio
import edge_tts
from pydub import AudioSegment
from moviepy.editor import TextClip, AudioFileClip, ColorClip, CompositeVideoClip, concatenate_videoclips
import os
import re
import shutil
from google.colab import files

# --- НАСТРОЙКИ ---
# Голоса для озвучки приключений Ли и пришельца [3]
VOICE_RU = "ru-RU-DmitryNeural"
VOICE_EN = "en-US-GuyNeural"
OUTPUT_FOLDER = "simple_karaoke"
EN_FILE = "original_en.txt"       # Английский текст из источников [2-5]
RU_FILE = "translated_ru.txt"     # Ваш перевод из источников [1, 6-13]

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

async def generate_voice(text, voice, filename):
    """Синтез речи через edge-tts"""
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(filename)

def create_text_video(text, audio_path, color):
    """Создание видео-фрагмента: текст на однотонном фоне"""
    audio = AudioFileClip(audio_path)
    # Создаем фон (темно-синий, как космос, который видел Ли [4])
    bg = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(audio.duration)
    # Наложение текста
    txt = TextClip(text, fontsize=60, color=color, size=(1100, None),
                   method='caption', font='Arial').set_duration(audio.duration).set_position('center')
    return CompositeVideoClip([bg, txt]).set_audio(audio)

async def main():
    # Проверка наличия ваших файлов в Colab
    if not os.path.exists(EN_FILE) or not os.path.exists(RU_FILE):
        print(f"Загрузите {EN_FILE} и {RU_FILE} в боковую панель!")
        return

    # Загрузка текстов из источников
    with open(EN_FILE, 'r', encoding='utf-8') as f:
        en_sents = re.split(r'(?<=[.!?])\s+', f.read().replace('\n', ' ').strip())
    with open(RU_FILE, 'r', encoding='utf-8') as f:
        ru_sents = re.split(r'(?<=[.!?])\s+', f.read().replace('\n', ' ').strip())

    for i, (en, ru) in enumerate(zip(en_sents, ru_sents)):
        temp_ru = f"ru_{i}.mp3"
        temp_en = f"en_{i}.mp3"
        output_mp4 = f"{OUTPUT_FOLDER}/{i+1:03d}.mp4"

        # 1. Озвучка
        await asyncio.gather(
            # generate_voice(ru.strip(), VOICE_RU, temp_ru),
            generate_voice(en.strip(), VOICE_EN, temp_en),
            generate_voice(en.strip(), VOICE_EN, temp_en)
        )

        # 2. Создание видео-частей
        # Желтый для перевода (например, "Они усердно трудились" [1])
        clip_ru = create_text_video(ru.strip(), temp_ru, 'yellow')
        # Голубой для оригинала (например, "They had worked hard" [2])
        clip_en = create_text_video(en.strip(), temp_en, 'cyan')

        # 3. Сборка финального видео для одного предложения
        final_video = concatenate_videoclips([clip_ru, clip_en])
        final_video.write_videofile(output_mp4, fps=24, codec='libx264', audio_codec='aac')

        # Очистка временных аудио
        os.remove(temp_ru)
        os.remove(temp_en)
        print(f"Видео {i+1:03d} создано.")

    # Шаг 3: Архивация и автоматическое скачивание
    shutil.make_archive('karaoke_simple', 'zip', OUTPUT_FOLDER)
    files.download('karaoke_simple.zip')
    print("Архив готов к скачиванию!")

# Запуск в Colab
await main()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


OSError: MoviePy error: the file ru_0.mp3 could not be found!
Please check that you entered the correct path.

In [24]:
await main()

OSError: MoviePy error: the file ru_0.mp3 could not be found!
Please check that you entered the correct path.

In [ ]:
import os

output_folder = 'simple_karaoke'
zip_file = 'karaoke_simple.zip'

if os.path.exists(output_folder):
    print(f"Содержимое папки '{output_folder}':")
    for item in os.listdir(output_folder):
        print(f"- {item}")
else:
    print(f"Папка '{output_folder}' не найдена.")

if os.path.exists(zip_file):
    print(f"\nАрхив '{zip_file}' успешно создан.")
else:
    print(f"\nАрхив '{zip_file}' не найден. Возможно, скрипт не завершился до конца.")

In [ ]:
import shutil
import os

folder_to_delete = 'simple_karaoke3' # Замените на имя папки, которую вы хотите удалить

if os.path.exists(folder_to_delete):
    shutil.rmtree(folder_to_delete)
    print(f"Папка '{folder_to_delete}' успешно удалена.")
elif folder_to_delete == 'simple_karaoke3':
    print("Пожалуйста, замените 'simple_karaoke3' на фактическое имя папки, которую вы хотите удалить.")
else:
    print(f"Папка '{folder_to_delete}' не найдена.")


Папка 'simple_karaoke3' успешно удалена.


In [ ]:
await main()

Moviepy - Building video simple_karaoke/001.mp4.
MoviePy - Writing audio in 001TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video simple_karaoke/001.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/001.mp4
Видео 001 создано.
Moviepy - Building video simple_karaoke/002.mp4.
MoviePy - Writing audio in 002TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video simple_karaoke/002.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/002.mp4
Видео 002 создано.
Moviepy - Building video simple_karaoke/003.mp4.
MoviePy - Writing audio in 003TEMP_MPY_wvf_snd.mp4


...

MoviePy - Done.
Moviepy - Writing video simple_karaoke/024.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/024.mp4
Видео 024 создано.
Moviepy - Building video simple_karaoke/025.mp4.
MoviePy - Writing audio in 025TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video simple_karaoke/025.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/025.mp4
Видео 025 создано.


WSServerHandshakeError: 503, message='Invalid response status', url='wss://speech.platform.bing.com/consumer/speech/synthesize/readaloud/edge/v1?TrustedClientToken=6A5AA1D4EAFF4E9FB37E23D68491D6F4&ConnectionId=88025ae46bcd4ad8a219595277c2c20a&Sec-MS-GEC=EF4C8921C9587BCD1AF6ED72E3155F97BF7B18E8024AABB44587790B8FC15897&Sec-MS-GEC-Version=1-143.0.3650.75'

In [ ]:
await main()

OSError: MoviePy Error: creation of None failed because of the following error:

convert-im6.q16: attempt to perform an operation not allowed by the security policy `@/tmp/tmpduvydjs6.txt' @ error/property.c/InterpretImageProperties/3706.
convert-im6.q16: no images defined `PNG32:/tmp/tmpyg5dwd3n.png' @ error/convert.c/ConvertImageCommand/3229.
.

.This error can be due to the fact that ImageMagick is not installed on your computer, or (for Windows users) that you didn't specify the path to the ImageMagick binary in file conf.py, or that the path you specified is incorrect

In [ ]:
await main()

OSError: MoviePy Error: creation of None failed because of the following error:

convert-im6.q16: attempt to perform an operation not allowed by the security policy `@/tmp/tmp0yf_f58p.txt' @ error/property.c/InterpretImageProperties/3706.
convert-im6.q16: no images defined `PNG32:/tmp/tmph57fquy3.png' @ error/convert.c/ConvertImageCommand/3229.
.

.This error can be due to the fact that ImageMagick is not installed on your computer, or (for Windows users) that you didn't specify the path to the ImageMagick binary in file conf.py, or that the path you specified is incorrect

In [ ]:
import os

en_file_exists = os.path.exists('original_en.txt')
ru_file_exists = os.path.exists('translated_ru.txt')

if en_file_exists and ru_file_exists:
    print("Файлы 'original_en.txt' и 'translated_ru.txt' успешно загружены.")
elif en_file_exists:
    print("Файл 'original_en.txt' загружен, но 'translated_ru.txt' отсутствует.")
elif ru_file_exists:
    print("Файл 'translated_ru.txt' загружен, но 'original_en.txt' отсутствует.")
else:
    print("Ни один из файлов ('original_en.txt', 'translated_ru.txt') не найден.")

Файлы 'original_en.txt' и 'translated_ru.txt' успешно загружены.


In [ ]:
# Изменение политики ImageMagick для разрешения создания временных файлов
# Эта команда пытается установить все права доступа для всех политик ImageMagick.
# Это должно решить проблему с 'security policy'.
!sudo sed -i 's/rights="none"/rights="all"/g' /etc/ImageMagick-6/policy.xml

print("Политика ImageMagick изменена. Теперь попробуйте повторно запустить основной скрипт.")

Политика ImageMagick изменена. Теперь попробуйте повторно запустить основной скрипт.


In [ ]:
from google.colab import files

print("Пожалуйста, загрузите 'original_en.txt':")
files.upload()

print("Пожалуйста, загрузите 'translated_ru.txt':")
files.upload()

Пожалуйста, загрузите 'original_en.txt':


Saving original_en.txt to original_en.txt
Saving translated_ru.txt to translated_ru.txt
Пожалуйста, загрузите 'translated_ru.txt':


Saving background.mp3 to background.mp3


{'background.mp3': b'ID3\x03\x00\x00\x00\x00@\x0eTYER\x00\x00\x00\x06\x00\x00\x002016\x00TDAT\x00\x00\x00\x06\x00\x00\x001406\x00TIME\x00\x00\x00\x06\x00\x00\x001221\x00PRIV\x00\x00\x17\xd4\x00\x00XMP\x00<?xpacket begin="\xef\xbb\xbf" id="W5M0MpCehiHzreSzNTczkc9d"?>\n<x:xmpmeta xmlns:x="adobe:ns:meta/" x:xmptk="Adobe XMP Core 5.6-c111 79.158325, 2015/09/10-01:10:20        ">\n <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">\n  <rdf:Description rdf:about=""\n    xmlns:xmpMM="http://ns.adobe.com/xap/1.0/mm/"\n    xmlns:stEvt="http://ns.adobe.com/xap/1.0/sType/ResourceEvent#"\n    xmlns:stRef="http://ns.adobe.com/xap/1.0/sType/ResourceRef#"\n    xmlns:xmp="http://ns.adobe.com/xap/1.0/"\n    xmlns:xmpDM="http://ns.adobe.com/xmp/1.0/DynamicMedia/"\n    xmlns:dc="http://purl.org/dc/elements/1.1/"\n    xmlns:bext="http://ns.adobe.com/bwf/bext/1.0/"\n   xmpMM:InstanceID="xmp.iid:5194885f-965a-4e3c-9804-d8a4f80dfbb3"\n   xmpMM:DocumentID="46ab6a50-cc00-c059-46b1-20b50000009b"\